# Create Three Continual-Learning Stages for 2010–2018

This notebook divides the simplified 2010–2018 labels into three chronological
continual-learning stages:

- Stage 1: 2010–2012
- Stage 2: 2013–2014
- Stage 3: 2015–2018

For each stage:

- The first 15% is reserved as an untouched chronological holdout.
- The remaining data are used for training.
- A 24-hour purge gap separates the holdout and training data.
- Another boundary gap prevents overlap with the following stage.

The files named `Stage*_test.csv` are internal stage holdouts used to measure
catastrophic forgetting. They are not the final 2025–2026 test set.

In [1]:
from pathlib import Path
from math import ceil

import pandas as pd

In [2]:
def find_project_root(start_path):
    """Find the repository root from the current notebook location."""

    for folder in [start_path, *start_path.parents]:
        if (folder / "data_labeling").is_dir() and (folder / "modeling").is_dir():
            return folder

    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

PROJECT_ROOT

PosixPath('/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-')

In [3]:
INPUT_FILE = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "simplified_data_labels"
    / "labels_2010_2018_binary.csv"
)

OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "continual_stages"
)

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_FILE)
print("Output folder:", OUTPUT_FOLDER)

Project root: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Input file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/simplified_data_labels/labels_2010_2018_binary.csv
Output folder: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages


In [4]:
labels = pd.read_csv(INPUT_FILE)

required_columns = {"label", "goes_class"}
missing_columns = required_columns - set(labels.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if labels["label"].isna().any():
    raise ValueError("Some image paths are missing.")

if labels["goes_class"].isna().any():
    raise ValueError("Some binary labels are missing.")

if not labels["goes_class"].isin([0, 1]).all():
    raise ValueError("The goes_class column must contain only 0 and 1.")

if labels["label"].duplicated().any():
    raise ValueError("Duplicate image paths were found.")

print("Total rows:", len(labels))
print()
print(labels["goes_class"].value_counts().sort_index())

labels.head()

Total rows: 63285

goes_class
0    54310
1     8975
Name: count, dtype: int64


,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0


In [5]:
timestamp_text = labels["label"].str.extract(
    r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})\.jpg$",
    expand=False,
)

if timestamp_text.isna().any():
    invalid_paths = labels.loc[timestamp_text.isna(), "label"].head()
    raise ValueError(
        f"Could not extract timestamps from these paths:\n{invalid_paths}"
    )

labels = labels.copy()

labels["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y.%m.%d_%H.%M.%S",
)

labels["year"] = labels["timestamp"].dt.year

labels = labels.sort_values("timestamp").reset_index(drop=True)

print("First timestamp:", labels["timestamp"].min())
print("Last timestamp:", labels["timestamp"].max())

labels.head()

First timestamp: 2010-12-06 07:00:00
Last timestamp: 2018-12-30 23:00:00


,label,goes_class,timestamp,year
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0,2010-12-06 07:00:00,2010
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0,2010-12-06 08:00:00,2010
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0,2010-12-06 09:00:00,2010
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0,2010-12-06 10:00:00,2010
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0,2010-12-06 11:00:00,2010


In [ ]:
STAGE_PERIODS = {
    1: (2010, 2012),
    2: (2013, 2014),
    3: (2015, 2018),
}

HOLDOUT_FRACTION = 0.15
PURGE_HOURS = 24

"""
Purged means intentionally removed from both training and testing to prevent data leakage.
Each image’s label depends on solar flares occurring during the following 24 hours. 
Therefore, images close to the train–holdout boundary can use overlapping flare events.
"""

stage_definition = pd.DataFrame(
    [
        {
            "Stage": stage_number,
            "Start year": start_year,
            "End year": end_year,
        }
        for stage_number, (start_year, end_year)
        in STAGE_PERIODS.items()
    ]
)

stage_definition

,Stage,Start year,End year
0,1,2010,2012
1,2,2013,2014
2,3,2015,2018


## Holdout design

The first 15% of every stage is used as a contiguous chronological holdout.

A final-period holdout was not selected because the end of Stage 3 falls near
the solar minimum and contains no FL examples. A holdout without both classes
cannot be evaluated reliably using TSS or HSS.

Using the first 15% provides a chronological block containing both FL and NF
samples for every stage while preserving most observations for training.
The holdouts remain naturally imbalanced and are not augmented or oversampled.

In [7]:
stage_files = {}
summary_rows = []

for stage_number, (start_year, end_year) in STAGE_PERIODS.items():

    # Select all observations belonging to this stage.
    stage_data = labels[
        labels["year"].between(start_year, end_year)
    ].copy()

    stage_data = stage_data.sort_values("timestamp").reset_index(drop=True)

    # Reserve the first 15% as the chronological stage holdout.
    holdout_size = ceil(len(stage_data) * HOLDOUT_FRACTION)

    stage_test = stage_data.iloc[:holdout_size].copy()

    # Training begins more than 24 hours after the last holdout image.
    holdout_end = stage_test["timestamp"].max()

    training_start_boundary = (
        holdout_end + pd.Timedelta(hours=PURGE_HOURS)
    )

    # Stop training at least 24 hours before the following calendar stage.
    next_period_start = pd.Timestamp(
        year=end_year + 1,
        month=1,
        day=1,
    )

    training_end_boundary = (
        next_period_start - pd.Timedelta(hours=PURGE_HOURS)
    )

    stage_train = stage_data[
        (stage_data["timestamp"] > training_start_boundary)
        & (stage_data["timestamp"] < training_end_boundary)
    ].copy()

    # Count rows intentionally excluded by the purge boundaries.
    purged_rows = len(stage_data) - len(stage_train) - len(stage_test)

    # Keep only the two columns required by the training pipeline.
    stage_train = stage_train[["label", "goes_class"]].reset_index(drop=True)
    stage_test = stage_test[["label", "goes_class"]].reset_index(drop=True)

    stage_files[stage_number] = {
        "train": stage_train,
        "test": stage_test,
    }

    summary_rows.append(
        {
            "Stage": stage_number,
            "Years": f"{start_year}–{end_year}",
            "Original": len(stage_data),
            "Train": len(stage_train),
            "Train NF": int((stage_train["goes_class"] == 0).sum()),
            "Train FL": int((stage_train["goes_class"] == 1).sum()),
            "Holdout": len(stage_test),
            "Holdout NF": int((stage_test["goes_class"] == 0).sum()),
            "Holdout FL": int((stage_test["goes_class"] == 1).sum()),
            "Purged": purged_rows,
        }
    )

stage_summary = pd.DataFrame(summary_rows)

stage_summary

,Stage,Years,Original,Train,Train NF,Train FL,Holdout,Holdout NF,Holdout FL,Purged
0,1,2010–2012,13606,11520,9181,2339,2041,1791,250,45
1,2,2013–2014,16061,13603,9680,3923,2410,2195,215,48
2,3,2015–2018,33618,28551,27319,1232,5043,4031,1012,24


Approximately 24 hours are removed after the holdout, and up to 24 hours are removed at the end of the stage.

Stage 3 has only 24 purged images because these are removed between its holdout and training sets. No additional end-of-stage purge is required because the final available image is from December 30, 2018 at 23:00, already 25 hours before the next period begins on January 1, 2019. This existing gap naturally prevents prediction-window overlap.

In [8]:
date_rows = []

for stage_number, files in stage_files.items():

    train_data = files["train"]
    test_data = files["test"]

    train_times = pd.to_datetime(
        train_data["label"].str.extract(
            r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})",
            expand=False,
        ),
        format="%Y.%m.%d_%H.%M.%S",
    )

    test_times = pd.to_datetime(
        test_data["label"].str.extract(
            r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})",
            expand=False,
        ),
        format="%Y.%m.%d_%H.%M.%S",
    )

    date_rows.append(
        {
            "Stage": stage_number,
            "Holdout start": test_times.min(),
            "Holdout end": test_times.max(),
            "Training start": train_times.min(),
            "Training end": train_times.max(),
        }
    )

stage_dates = pd.DataFrame(date_rows)

stage_dates

,Stage,Holdout start,Holdout end,Training start,Training end
0,1,2010-12-06 07:00:00,2011-03-21 08:00:00,2011-03-22 10:00:00,2012-12-30 23:00:00
1,2,2013-01-01 00:00:00,2013-04-19 00:00:00,2013-04-20 01:00:00,2014-12-30 23:00:00
2,3,2015-01-01 00:00:00,2015-08-04 17:00:00,2015-08-05 18:00:00,2018-12-30 23:00:00


In [10]:
for _, row in stage_dates.iterrows():

    internal_gap = (
        row["Training start"] - row["Holdout end"]
    ).total_seconds() / 3600

    print(
        f"Stage {row['Stage']}: "
        f"{internal_gap:.0f} hours between the last holdout image "
        f"and the first training image"
    )

    assert internal_gap > PURGE_HOURS

print("\nAll stage holdout/training boundaries have a 24-hour purge.")

Stage 1: 26 hours between the last holdout image and the first training image
Stage 2: 25 hours between the last holdout image and the first training image
Stage 3: 25 hours between the last holdout image and the first training image

All stage holdout/training boundaries have a 24-hour purge.


In [11]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

for stage_number, files in stage_files.items():

    train_file = OUTPUT_FOLDER / f"Stage{stage_number}_train.csv"
    test_file = OUTPUT_FOLDER / f"Stage{stage_number}_holdout.csv"

    files["train"].to_csv(train_file, index=False)
    files["test"].to_csv(test_file, index=False)

    print("Created:", train_file)
    print("Created:", test_file)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages/Stage1_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages/Stage1_holdout.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages/Stage2_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages/Stage2_holdout.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages/Stage3_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages/Stage3_holdout.csv
